In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../train.csv")

X = df.drop(columns=["Price_INR_Lakhs", "Property_ID"])
y = df["Price_INR_Lakhs"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42
)

categorical_features = [
    "City", "Locality_Type", "Property_Type", "Furnishing_Status"
]

numerical_features = [
    "BHK", "Bathrooms", "Super_Area_SqFt", "Carpet_Area_SqFt",
    "Floor_Number", "Total_Floors", "Age_of_Property",
    "Parking", "Lift_Available", "Gated_Community",
    "Distance_to_Metro_km", "Distance_to_City_Center_km"
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)],
    sparse_threshold=0
)

### Log-transforming the target

House prices are right-skewed: a few very expensive properties pull the mean up.
Training on `log1p(price)` and converting predictions back with `expm1` often helps tree models
on skewed targets.

In [2]:
print(y.describe().round(2))
print(f"\nSkewness: {y.skew():.2f}")

count    80000.00
mean       142.61
std        147.24
min          2.77
25%         54.35
50%         96.60
75%        176.71
max       3334.92
Name: Price_INR_Lakhs, dtype: float64

Skewness: 3.66


In [3]:
import numpy as np
from sklearn.compose import TransformedTargetRegressor

tuned_params = {
    "learning_rate": 0.05,
    "max_iter": 500,
    "max_leaf_nodes": 15,
    "min_samples_leaf": 100,
    "l2_regularization": 1.0
}

models = {
    "raw target": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", HistGradientBoostingRegressor(random_state=42, **tuned_params))
    ]),
    "log target": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", TransformedTargetRegressor(
            regressor=HistGradientBoostingRegressor(random_state=42, **tuned_params),
            func=np.log1p,
            inverse_func=np.expm1
        ))
    ])
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    results.append({
        "target": name,
        "MAE": mean_absolute_error(y_val, y_pred),
        "RMSE": mean_squared_error(y_val, y_pred) ** 0.5,
        "R2": r2_score(y_val, y_pred)
    })

pd.DataFrame(results)

,target,MAE,RMSE,R2
0,raw target,32.584237,68.062936,0.790761
1,log target,32.257635,68.985363,0.785051


Mixed result: the log target lowers MAE (better on typical houses) but raises RMSE and lowers R2,
because it under-predicts the very expensive properties. Since we evaluate on R2, we keep the raw target.